In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=False)
context

# FLenQA lens drift as prompts grow

This notebook asks which tokens become more or less prominent at the lens's `final_prompt` position as prompt length grows. Token prominence is **reciprocal rank** in the saved top-25; a token outside the top-25 contributes zero. This is a descriptive top-25 analysis, not the model's full probability distribution.

The cells are intentionally small: paths, loading, aggregation, metrics, and plots are separate so intermediate analysis can be inserted easily.

In [ ]:
import string

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from experiments.jlens_readout_sanity.constants import MODEL_PATH

FULL_RUN = context.runs_dir / "flenqa-full-run"
ACCURACY_PATH = context.runs_dir / "flenqa-accuracy" / "results.parquet"
FULL_RUN, ACCURACY_PATH

In [ ]:
def require_same_prompt_ids(reference, **tables):
    reference = set(reference)
    for name, values in tables.items():
        values = set(values)
        missing = len(reference - values)
        extra = len(values - reference)
        if missing or extra:
            raise ValueError(
                f"{name} has {missing} missing and {extra} extra prompt IDs"
            )

In [ ]:
accuracy = pd.read_parquet(
    ACCURACY_PATH,
    columns=["prompt_id", "ctx_size", "n_input_tokens", "correct"],
)
display(accuracy.head())
display(accuracy.groupby("ctx_size")["n_input_tokens"].agg(["min", "median", "max"]))
accuracy.shape

In [ ]:
prompts = pd.read_parquet(FULL_RUN / "prompts", columns=["prompt_id", "task"])
positions = pd.read_parquet(
    FULL_RUN / "positions", columns=["prompt_id", "position", "label"]
)
final_positions = (
    positions[positions["label"] == "final_prompt"]
    .drop(columns="label")
    .rename(columns={"position": "final_position"})
)

require_same_prompt_ids(
    prompts["prompt_id"],
    accuracy=accuracy["prompt_id"],
    final_positions=final_positions["prompt_id"],
)
prompt_info = prompts.merge(accuracy, on="prompt_id", validate="one_to_one").merge(
    final_positions, on="prompt_id", validate="one_to_one"
)
display(prompt_info.head())
prompt_info.groupby("ctx_size")["prompt_id"].size()

In [ ]:
def summarize_tokens(topk, prompt_info):
    """Summarize final-position tokens and return the prompts observed."""
    rows = topk.merge(
        prompt_info, on="prompt_id", how="left", validate="many_to_one", indicator=True
    )
    if (rows["_merge"] != "both").any():
        raise ValueError("top-k rows reference unknown prompt IDs")
    rows = rows[rows["position"] == rows["final_position"]].copy()
    rows["rank_score"] = 1 / rows["rank"]

    keys = ["lens_kind", "layer", "ctx_size", "token_id"]
    summary = rows.groupby(keys, as_index=False).agg(
        appearances=("token_id", "size"),
        rank_score=("rank_score", "sum"),
        logit_sum=("logit", "sum"),
    )
    return summary, set(rows["prompt_id"])

In [ ]:
def combine_token_summaries(summaries):
    keys = ["lens_kind", "layer", "ctx_size", "token_id"]
    return (
        pd.concat(summaries)
        .groupby(keys, as_index=False)[["appearances", "rank_score", "logit_sum"]]
        .sum()
    )

In [ ]:
def measure_distribution_drift(token_stats):
    """Compare each length's token distribution with the shortest length."""
    baseline = token_stats["ctx_size"].min()
    records = []

    for (lens_kind, layer), rows in token_stats.groupby(["lens_kind", "layer"]):
        scores = rows.pivot_table(
            index="token_id", columns="ctx_size", values="rank_score", fill_value=0
        )
        shares = scores / scores.sum()
        distances = 0.5 * shares.sub(shares[baseline], axis=0).abs().sum()
        records.extend(
            {
                "lens_kind": lens_kind,
                "layer": layer,
                "ctx_size": length,
                "total_variation": distance,
            }
            for length, distance in distances.items()
        )

    return pd.DataFrame(records)

## Summarize the large top-k table

The Parquet files are read in bounded batches. Each shard is reduced before the next one is read, so raw top-k rows do not accumulate in memory.

In [ ]:
topk_columns = [
    "prompt_id",
    "lens_kind",
    "layer",
    "position",
    "rank",
    "token_id",
    "logit",
]
topk_files = sorted((FULL_RUN / "topk").glob("*.parquet"))
if not topk_files:
    raise FileNotFoundError(f"No top-k shards found in {FULL_RUN / 'topk'}")

prompt_keys = prompt_info[["prompt_id", "ctx_size", "final_position"]]
seen_topk_ids = set()
token_stats = None
for path in tqdm(topk_files, desc="Reading top-k shards"):
    batch_summaries = []
    batches = pq.ParquetFile(path).iter_batches(
        batch_size=250_000, columns=topk_columns
    )
    for batch in batches:
        topk = batch.to_pandas()
        summary, batch_ids = summarize_tokens(topk, prompt_keys)
        batch_summaries.append(summary)
        seen_topk_ids.update(batch_ids)

    shard_summary = combine_token_summaries(batch_summaries)
    token_stats = (
        shard_summary
        if token_stats is None
        else combine_token_summaries([token_stats, shard_summary])
    )

require_same_prompt_ids(prompt_info["prompt_id"], topk=seen_topk_ids)
del batch, batch_summaries, batches, shard_summary, summary, topk

In [ ]:
prompt_counts = (
    prompt_info.groupby("ctx_size", as_index=False)["prompt_id"]
    .nunique()
    .rename(columns={"prompt_id": "prompt_count"})
)
token_stats = token_stats.merge(prompt_counts, on="ctx_size")
token_stats["prominence"] = token_stats["rank_score"] / token_stats["prompt_count"]
token_stats["visibility"] = token_stats["appearances"] / token_stats["prompt_count"]
token_stats["mean_visible_logit"] = (
    token_stats["logit_sum"] / token_stats["appearances"]
)
display(token_stats.head())
token_stats.shape

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
token_text = {
    token_id: tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
    for token_id in token_stats["token_id"].unique()
}
COMMON_STOPWORDS = set(
    "a an and are as at be by for from in is it of on or that the this to was were with".split()
)


def token_group(text):
    value = text.strip().lower()
    if not value:
        return "whitespace"
    if value in COMMON_STOPWORDS:
        return "common stopword"
    if value.isalpha():
        return "word"
    if value.isdigit():
        return "number"
    if all(character in string.punctuation for character in value):
        return "punctuation"
    return "other"


token_stats["token"] = token_stats["token_id"].map(token_text)
token_stats["token_group"] = token_stats["token"].map(token_group)

## General top-25 drift

Total variation is 0 when the normalized top-25 reciprocal-rank mass matches the shortest prompts and approaches 1 as it becomes completely different.

In [ ]:
drift = measure_distribution_drift(token_stats)
lens_kinds = sorted(drift["lens_kind"].unique())
fig, axes = plt.subplots(1, len(lens_kinds), figsize=(13, 5), sharey=True)

for axis, lens_kind in zip(axes, lens_kinds, strict=True):
    matrix = drift[drift["lens_kind"] == lens_kind].pivot(
        index="layer", columns="ctx_size", values="total_variation"
    )
    image = axis.imshow(matrix, aspect="auto", origin="lower", cmap="magma")
    axis.set_xticks(range(len(matrix.columns)), matrix.columns)
    axis.set_yticks(range(len(matrix.index)), matrix.index)
    axis.set_title(f"{lens_kind.title()} Lens")
    axis.set_xlabel("Nominal prompt length")
    fig.colorbar(image, ax=axis, label="Top-25 rank-mass drift")

axes[0].set_ylabel("Layer")
fig.suptitle("How much top-25 reciprocal-rank mass changes with prompt length")
plt.tight_layout()
plt.show()

## Which tokens changed?

Choose a lens and layer here. The default uses the final Jacobian Lens layer; edit these two values to explore another slice.

In [ ]:
LENS_KIND = "jacobian"
LAYER = int(token_stats["layer"].max())

selected = token_stats.query("lens_kind == @LENS_KIND and layer == @LAYER")
SHORT_LENGTH = int(selected["ctx_size"].min())
LONG_LENGTH = int(selected["ctx_size"].max())
LENS_KIND, LAYER, SHORT_LENGTH, LONG_LENGTH

In [ ]:
prominence = selected.pivot_table(
    index="token_id", columns="ctx_size", values="prominence", fill_value=0
)
visibility = selected.pivot_table(
    index="token_id", columns="ctx_size", values="visibility", fill_value=0
)
visible_logit = selected.pivot_table(
    index="token_id", columns="ctx_size", values="mean_visible_logit"
)

changes = pd.DataFrame(index=prominence.index)
changes["token"] = changes.index.map(token_text)
changes["token_group"] = changes["token"].map(token_group)
changes["short_prominence"] = prominence[SHORT_LENGTH]
changes["long_prominence"] = prominence[LONG_LENGTH]
changes["prominence_change"] = prominence[LONG_LENGTH] - prominence[SHORT_LENGTH]
changes["visibility_change"] = visibility[LONG_LENGTH] - visibility[SHORT_LENGTH]
changes["conditional_logit_change"] = (
    visible_logit[LONG_LENGTH] - visible_logit[SHORT_LENGTH]
)
risers = changes.nlargest(15, "prominence_change").reset_index()
fallers = changes.nsmallest(15, "prominence_change").reset_index()

In [ ]:
columns = [
    "token_id",
    "token",
    "token_group",
    "short_prominence",
    "long_prominence",
    "prominence_change",
    "visibility_change",
    "conditional_logit_change",
]
print("Logit change is conditional on top-25 visibility at both endpoint lengths.")
display("Tokens rising with prompt length", risers[columns])
display("Tokens falling with prompt length", fallers[columns])

In [ ]:
interesting_ids = [*risers["token_id"].head(4), *fallers["token_id"].head(4)]
trends = selected[selected["token_id"].isin(interesting_ids)].pivot_table(
    index="ctx_size", columns="token_id", values="prominence", fill_value=0
)

plt.figure(figsize=(10, 5))
for token_id in trends.columns:
    plt.plot(
        trends.index, trends[token_id], marker="o", label=repr(token_text[token_id])
    )
plt.xlabel("Nominal prompt length")
plt.ylabel("Mean reciprocal-rank prominence")
plt.title(f"Largest token changes — {LENS_KIND} lens, layer {LAYER}")
plt.legend(ncol=2)
plt.grid(alpha=0.25)
plt.show()

In [ ]:
group_scores = selected.pivot_table(
    index="ctx_size",
    columns="token_group",
    values="rank_score",
    aggfunc="sum",
    fill_value=0,
)
group_share = group_scores.div(group_scores.sum(axis=1), axis=0)
display(group_share)
group_share.plot(figsize=(10, 5), marker="o")
plt.xlabel("Nominal prompt length")
plt.ylabel("Share of top-k rank mass")
plt.title(f"Token groups — {LENS_KIND} lens, layer {LAYER}")
plt.grid(alpha=0.25)
plt.show()

## Put drift beside model accuracy

Accuracy is context only—it is not part of the drift definition.

In [ ]:
accuracy_by_length = (
    prompt_info.groupby("ctx_size", as_index=False)["correct"]
    .mean()
    .rename(columns={"correct": "accuracy"})
)
mean_drift = drift.groupby(["lens_kind", "ctx_size"], as_index=False)[
    "total_variation"
].mean()
accuracy_context = mean_drift.merge(accuracy_by_length, on="ctx_size")
display(accuracy_context)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(accuracy_by_length["ctx_size"], accuracy_by_length["accuracy"], marker="o")
axes[0].set(title="Unique-prompt accuracy", xlabel="Prompt length", ylabel="Accuracy")
for lens_kind, rows in mean_drift.groupby("lens_kind"):
    axes[1].plot(rows["ctx_size"], rows["total_variation"], marker="o", label=lens_kind)
axes[1].set(
    title="Mean drift across layers",
    xlabel="Prompt length",
    ylabel="Top-25 rank-mass drift",
)
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
long_drift = drift.query("lens_kind == @LENS_KIND and ctx_size == @LONG_LENGTH")
most_changed_layer = long_drift.loc[long_drift["total_variation"].idxmax()]
strongest_riser = risers.iloc[0]
strongest_faller = fallers.iloc[0]

print(
    f"Largest {LENS_KIND} top-25 rank-mass drift at length {LONG_LENGTH}: "
    f"layer {int(most_changed_layer.layer)} "
    f"(total variation {most_changed_layer.total_variation:.3f})."
)
print(
    f"At selected layer {LAYER}, strongest riser: {strongest_riser.token!r} "
    f"({strongest_riser.prominence_change:+.4f} prominence)."
)
print(
    f"At selected layer {LAYER}, strongest faller: {strongest_faller.token!r} "
    f"({strongest_faller.prominence_change:+.4f} prominence)."
)